In [1]:
from pathlib import Path
import os, sys, json, hashlib, importlib.util
import numpy as np
import pandas as pd
import scipy.sparse as sp

ROOT = Path("/home/featurize/work/biological_prior_validation").resolve()
CONFIG = {
    "project_root": str(ROOT),
    "run_path": str(ROOT / "data/splits/step2_q05_seed20261008"),
    "development_name": "inner_q05_seed20261008",
    "inner_q": 0.5,
    "seed": 20261008,
    "inner_n_genes": 2500,
}
if not ROOT.is_dir():
    raise FileNotFoundError(f"Please change ROOT to the existing project directory：{ROOT}")
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))


{
  "project_root": "/home/featurize/work/biological_prior_validation",
  "run_path": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008",
  "development_name": "inner_q05_seed20261008",
  "inner_q": 0.5,
  "seed": 20261008,
  "inner_n_genes": 2500
}


In [2]:
SOURCE = '"""Step 4: within-replicate nested molecular development; no final B access."""\nfrom pathlib import Path\nimport hashlib, json, os, platform, subprocess, sys, shutil\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport scipy.sparse as sp\n\nLINES=[\'A172\',\'T98G\',\'U87MG\']\nDRUGS=[\'Vehicle\',\'Lapatinib\',\'Nintedanib\',\'Trametinib\',\'ZSTK474\']\nREPS=[\'replicate_1\',\'replicate_2\']\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n\ndef digest(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(1048576),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef read_json(p):return json.loads(Path(p).read_text())\n\ndef save_json(p,obj):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    temp=p.with_name(p.name+\'.tmp\')\n    temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False,default=str))\n    temp.replace(p)\n\ndef freeze_json(path,obj):\n    if Path(path).exists() and read_json(path)!=obj:\n        raise ValueError(f\'Frozen configuration changed: {path}; use a new development_name\')\n    save_json(path,obj)\n\ndef checked_counts(a):\n    a=sp.csr_matrix(a,copy=True);a.sum_duplicates();a.sort_indices();a.eliminate_zeros()\n    if not np.isfinite(a.data).all() or (a.data<0).any() or not np.equal(a.data,np.floor(a.data)).all():\n        raise ValueError(\'Counts must be finite nonnegative integers\')\n    if (a.data>np.iinfo(np.int64).max).any():raise ValueError(\'Counts overflow int64\')\n    return a.astype(np.int64)\n\ndef split_inner(a,p,seed,block):\n    if not np.isfinite(p) or not 0<p<1:raise ValueError(\'inner_q must be strictly between 0 and 1\')\n    if not isinstance(seed,(int,np.integer)) or seed<0 or block<0:raise ValueError(\'Invalid seed/block\')\n    a=checked_counts(a)\n    rng=np.random.Generator(np.random.PCG64(np.random.SeedSequence([int(seed),int(block)])))\n    train=a.copy();train.data=rng.binomial(a.data,p).astype(np.int64)\n    valid=a.copy();valid.data=a.data-train.data\n    train.eliminate_zeros();valid.eliminate_zeros()\n    if (train+valid-a).nnz:raise AssertionError(\'Inner split does not reconstruct A\')\n    return train,valid\n\ndef log_normalize(a,scale):\n    if not np.isfinite(scale) or scale<=0:raise ValueError(\'Invalid normalization scale\')\n    lib=np.asarray(a.sum(axis=1)).ravel().astype(np.int64)\n    factors=np.divide(scale,lib,out=np.zeros(len(lib),dtype=float),where=lib>0)\n    y=sp.diags(factors)@a.astype(float);y=y.tocsr();y.data=np.log1p(y.data)\n    return y,lib\n\ndef load_inputs(cfg):\n    root=Path(cfg[\'project_root\']).resolve();run=Path(cfg[\'run_path\']).resolve()\n    if not run.is_relative_to(root/\'data/splits\'):raise ValueError(\'run_path must be inside project data/splits\')\n    if not cfg[\'development_name\'] or Path(cfg[\'development_name\']).name!=cfg[\'development_name\'] or cfg[\'development_name\'] in [\'.\',\'..\']:\n        raise ValueError(\'development_name must be a simple directory name\')\n    if not np.isfinite(cfg[\'inner_q\']) or not 0<cfg[\'inner_q\']<1:raise ValueError(\'Invalid inner_q\')\n    if not isinstance(cfg[\'seed\'],int) or cfg[\'seed\']<0:raise ValueError(\'Invalid seed\')\n    if not isinstance(cfg[\'inner_n_genes\'],int) or cfg[\'inner_n_genes\']<1:raise ValueError(\'Invalid gene count\')\n    metric_path=run/\'evaluation/metric_contract.json\'\n    if (run/\'evaluation/target_access_lock.json\').exists():\n        raise ValueError(\'Final B was already accessed; do not start a new tuning protocol in this run\')\n    pair_path=run/\'A_preprocessing/evaluation_pair_contract.json\';pair=read_json(pair_path)\n    metric=read_json(metric_path);ident=read_json(run/\'identity.json\')\n    if metric[\'pair_contract_sha256\']!=digest(pair_path):raise ValueError(\'Step 2/3 contract mismatch\')\n    if metric[\'q\']!=pair[\'q\'] or pair[\'q\']!=ident[\'q\']:raise ValueError(\'Outer q mismatch\')\n    if metric[\'normalization_scale\']!=pair[\'normalization_scale\']:raise ValueError(\'Normalization mismatch\')\n    prep=run/\'A_preprocessing\';cp=prep/pair[\'cells_file\'];gp=prep/pair[\'genes_file\']\n    for path,key in [(cp,\'cells_sha256\'),(gp,\'genes_sha256\')]:\n        if digest(path)!=pair[key] or metric[key]!=pair[key]:raise ValueError(\'Frozen cells/genes changed\')\n    cells=pd.read_parquet(cp).sort_values(\'matrix_row\').reset_index(drop=True)\n    genes=pd.read_parquet(gp).sort_values(\'selected_col\').reset_index(drop=True)\n    if not np.array_equal(cells.matrix_row,np.arange(len(cells))) or not cells.cell_barcode.is_unique or cells.cell_barcode.isna().any():raise ValueError(\'Cell mapping mismatch\')\n    if not cells.evaluation_eligible.isin([True,False]).all():raise ValueError(\'Invalid evaluation mask\')\n    if not np.array_equal(genes.selected_col,np.arange(len(genes))) or not genes.gene_idx.is_unique:raise ValueError(\'Gene mapping mismatch\')\n    n=int(cells.evaluation_eligible.sum())\n    if metric.get(\'version\',\'step3-fixed-evaluation-v1\')!=\'step3-fixed-evaluation-v1\':raise ValueError(\'Unexpected metric contract version\')\n    if not np.isfinite(metric[\'mu_floor\']) or metric[\'mu_floor\']<=0:raise ValueError(\'Invalid fixed metric floor\')\n    for obj in [pair,metric]:\n        if (obj[\'n_cells\'],obj[\'n_genes\'],obj[\'n_pairs\'])!=(n,len(genes),n*len(genes)):raise ValueError(\'Pair totals mismatch\')\n    good=cells.loc[cells.evaluation_eligible]\n    for field,values in [(\'cell_line\',LINES),(\'drug\',DRUGS),(\'replicate\',REPS)]:\n        if not good[field].isin(values).all():raise ValueError(f\'Unknown eligible {field}\')\n    raw=pd.read_parquet(run/\'genes_all.parquet\').sort_values(\'gene_col\').reset_index(drop=True)\n    if not np.array_equal(raw.gene_col,np.arange(len(raw))) or not raw.gene_idx.is_unique:raise ValueError(\'Raw gene mapping mismatch\')\n    if not genes.gene_idx.isin(raw.gene_idx).all() or not np.array_equal(genes.gene_col.to_numpy(),genes.gene_idx.to_numpy()-1):raise ValueError(\'Outer gene IDs differ from raw mapping\')\n    mapping_path=root/\'data/metadata_audit/ppi_gene_mapping.csv\'\n    mapping=pd.read_csv(mapping_path,dtype={\'gene_id\':\'string\',\'gene_symbol\':\'string\'})\n    if not mapping.gene_idx.is_unique:raise ValueError(\'Ambiguous PPI mapping\')\n    joined=raw[[\'gene_idx\',\'gene_id\',\'gene_symbol\']].merge(mapping[[\'gene_idx\',\'gene_id\',\'gene_symbol\',\'ppi_edge_supported\']],on=\'gene_idx\',validate=\'one_to_one\',how=\'left\',suffixes=(\'\',\'_ppi\'))\n    for field in [\'gene_id\',\'gene_symbol\']:\n        if not joined[field].fillna(\'\').eq(joined[field+\'_ppi\'].fillna(\'\')).all():raise ValueError(\'PPI annotation IDs differ\')\n    if joined.ppi_edge_supported.isna().any():raise ValueError(\'Incomplete PPI mapping\')\n    support=joined.ppi_edge_supported.eq(True).to_numpy()\n    return root,run,cells,genes,raw,support,pair,metric,ident\n\ndef coverage(frame,mask,ng):\n    index=pd.MultiIndex.from_product([REPS,LINES,DRUGS],names=[\'replicate\',\'cell_line\',\'drug\'])\n    out=frame.loc[mask].groupby([\'replicate\',\'cell_line\',\'drug\'],observed=True).size().reindex(index,fill_value=0).rename(\'n_cells\').reset_index()\n    out[\'n_genes\']=ng;out[\'n_pairs\']=out.n_cells*ng\n    return out\n\ndef rank_genes(raw,support,stats,requested):\n    n,sums,squares,detected=stats\n    if n<2:raise ValueError(\'At least two positive-inner-training-library cells required per replicate\')\n    table=raw.copy();table[\'train_log_mean\']=sums/n\n    table[\'train_log_variance\']=np.maximum((squares-sums*sums/n)/(n-1),0)\n    table[\'train_detection_fraction\']=detected/n;table[\'ppi_supported\']=support\n    candidates=table.loc[support & table.gene_symbol.notna() & table.train_log_variance.gt(0)]\n    selected=candidates.sort_values([\'train_log_variance\',\'train_detection_fraction\',\'gene_idx\'],ascending=[False,False,True],kind=\'stable\').drop_duplicates(\'gene_symbol\').head(requested).copy()\n    if len(selected)!=requested:raise ValueError(f\'Only {len(selected)} variable PPI genes; requested {requested}. Do not silently reduce gene count.\')\n    selected[\'selected_col\']=np.arange(len(selected))\n    return selected,table\n\ndef save_environment(root,out):\n    try:\n        commit=subprocess.run([\'git\',\'-C\',str(root),\'rev-parse\',\'HEAD\'],text=True,capture_output=True,timeout=15)\n        git={\'commit\':commit.stdout.strip() if commit.returncode==0 else None,\'status\':\'available\' if commit.returncode==0 else \'not_a_git_checkout\'}\n        if commit.returncode==0:\n            git[\'dirty\']=bool(subprocess.run([\'git\',\'-C\',str(root),\'status\',\'--porcelain\'],text=True,capture_output=True,timeout=15).stdout.strip())\n    except (FileNotFoundError,subprocess.TimeoutExpired):git={\'commit\':None,\'status\':\'git_unavailable\'}\n    record={\'python\':sys.version,\'executable\':sys.executable,\'platform\':platform.platform(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'scipy\':scipy.__version__,\'engine_sha256\':ENGINE_SHA,\'git\':git}\n    save_json(out/\'environment.json\',record)\n    freeze=subprocess.run([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True,capture_output=True,timeout=60)\n    if freeze.returncode:raise RuntimeError(\'pip freeze failed\')\n    (out/\'python_requirements.lock.txt\').write_text(freeze.stdout)\n    return record\n\ndef prepare_development(cfg):\n    root,run,cells,genes,raw,support,pair,metric,ident=load_inputs(cfg)\n    out=run/\'development\'/cfg[\'development_name\'];out.mkdir(parents=True,exist_ok=True)\n    identity={\'version\':\'step4-within-replicate-nested-A-v1\',\'engine_sha256\':ENGINE_SHA,\'run_path\':str(run),\'inner_q\':cfg[\'inner_q\'],\'seed\':cfg[\'seed\'],\'inner_n_genes\':cfg[\'inner_n_genes\'],\n              \'pair_contract_sha256\':digest(run/\'A_preprocessing/evaluation_pair_contract.json\'),\'metric_contract_sha256\':digest(run/\'evaluation/metric_contract.json\'),\'raw_genes_sha256\':digest(run/\'genes_all.parquet\'),\n              \'ppi_mapping_sha256\':digest(root/\'data/metadata_audit/ppi_gene_mapping.csv\'),\'numpy\':np.__version__,\'scipy\':scipy.__version__,\'rng\':\'PCG64 SeedSequence([seed, block]); canonical CSR order; fixed original blocks\'}\n    freeze_json(out/\'identity.json\',identity)\n    size=int(ident[\'cells_per_block\']);ng=len(raw)\n    if size<1:raise ValueError(\'Invalid original block size\')\n    nblocks=(len(cells)+size-1)//size\n    stats={rep:[0,np.zeros(ng),np.zeros(ng),np.zeros(ng,dtype=np.int64)] for rep in REPS}\n    trainlib=np.zeros(len(cells),dtype=np.int64);validlib=np.zeros(len(cells),dtype=np.int64)\n    eligible=cells.evaluation_eligible.to_numpy(dtype=bool);pieces=[]\n    for block in range(nblocks):\n        folder=run/f\'block_{block:05d}\';source=folder/\'A_input.npz\';receipt=read_json(folder/\'receipt.json\')\n        expected=next(x[\'sha256\'] for x in receipt[\'files\'] if x[\'name\']==\'A_input.npz\')\n        if digest(source)!=expected:raise ValueError(\'A source SHA256 changed\')\n        start=block*size;stop=min(start+size,len(cells));target=out/f\'block_{block:05d}\'\n        if target.exists():\n            rec=read_json(target/\'receipt.json\')\n            if rec[\'A_source_sha256\']!=expected or rec[\'identity\']!=identity:raise ValueError(\'Cached inner split provenance changed\')\n            for f in rec[\'files\']:\n                if digest(target/f[\'name\'])!=f[\'sha256\']:raise ValueError(\'Cached inner split damaged\')\n            train=sp.load_npz(target/\'A_train.npz\');valid=sp.load_npz(target/\'A_validation.npz\')\n        else:\n            a=checked_counts(sp.load_npz(source))\n            if a.shape!=(stop-start,ng):raise ValueError(\'A block dimensions mismatch\')\n            train,valid=split_inner(a,cfg[\'inner_q\'],cfg[\'seed\'],block)\n            temp=out/f\'block_{block:05d}.partial\'\n            if temp.exists():shutil.rmtree(temp)\n            temp.mkdir();files=[]\n            for name,value in [(\'A_train.npz\',train),(\'A_validation.npz\',valid)]:\n                sp.save_npz(temp/name,value,compressed=True);files.append({\'name\':name,\'sha256\':digest(temp/name)})\n            rec={\'A_source_sha256\':expected,\'identity\':identity,\'files\':files}\n            save_json(temp/\'receipt.json\',rec);temp.replace(target)\n        if train.shape!=(stop-start,ng) or valid.shape!=train.shape:raise ValueError(\'Inner split dimensions mismatch\')\n        y,lt=log_normalize(train,pair[\'normalization_scale\']);lv=np.asarray(valid.sum(axis=1)).ravel().astype(np.int64)\n        if (eligible[start:stop] & ((lt+lv)<=0)).any():raise ValueError(\'Frozen outer-eligible cells must have positive A libraries\')\n        trainlib[start:stop]=lt;validlib[start:stop]=lv\n        for rep in REPS:\n            keep=eligible[start:stop] & cells.replicate.iloc[start:stop].eq(rep).to_numpy() & (lt>0)\n            sub=y[keep];count=train[keep];s=stats[rep];s[0]+=int(keep.sum())\n            s[1]+=np.asarray(sub.sum(axis=0)).ravel();s[2]+=np.asarray(sub.multiply(sub).sum(axis=0)).ravel();s[3]+=np.asarray((count>0).sum(axis=0)).ravel()\n        pieces.append({\'block\':block,\'rows\':stop-start,\'A_umi\':int(lt.sum())+int(lv.sum()),\'train_umi\':int(lt.sum()),\'validation_umi\':int(lv.sum()),\'train_zero_cells\':int((lt==0).sum()),\'validation_zero_cells\':int((lv==0).sum())})\n        print(f\'Step 4 inner split {block+1}/{nblocks}\',flush=True)\n    # Final pair membership is untouched; zero validation libraries are NEVER excluded.\n    inner_mask=eligible & (trainlib>0)\n    cell_roles=cells.copy();cell_roles[\'library_A_train\']=trainlib;cell_roles[\'library_A_validation\']=validlib\n    cell_roles[\'inner_evaluation_eligible\']=inner_mask\n    cell_roles[\'outer_role\']=np.where(eligible,\'A=fit/refit; B=final_target\',\'excluded_by_existing_Step2_contract\')\n    cell_roles[\'inner_role\']=np.where(inner_mask,\'A_train=model_input; A_validation=parameter_selection\',np.where(eligible,\'inner_train_zero; retain_outer_I\',\'outside_outer_I\'))\n    cell_roles.to_parquet(out/\'cell_roles.parquet\',index=False)\n    totals=pd.DataFrame(pieces);totals.to_csv(out/\'block_summary.csv\',index=False)\n    if not totals.A_umi.eq(totals.train_umi+totals.validation_umi).all():raise AssertionError(\'Library conservation failed\')\n    summaries=[]\n    for rep in REPS:\n        selected,table=rank_genes(raw,support,stats[rep],cfg[\'inner_n_genes\'])\n        selected.to_parquet(out/f\'{rep}_inner_selected_genes.parquet\',index=False)\n        table.to_parquet(out/f\'{rep}_inner_gene_stats.parquet\',index=False)\n        summaries.append({\'replicate\':rep,\'n_train_fit_cells\':int(stats[rep][0]),\'n_inner_genes\':len(selected),\'selected_genes_sha256\':digest(out/f\'{rep}_inner_selected_genes.parquet\')})\n    coverage(cell_roles,eligible,len(genes)).to_csv(out/\'outer_coverage.csv\',index=False)\n    coverage(cell_roles,inner_mask,cfg[\'inner_n_genes\']).to_csv(out/\'inner_coverage.csv\',index=False)\n    group=[\'replicate\',\'cell_line\',\'drug\',\'dose_raw\',\'sgRNA_raw\',\'target_raw\',\'hash_plate\',\'hash_label_well\',\'library_id\']\n    group=[k for k in group if k in cells]\n    fine=cell_roles.loc[eligible].groupby(group,dropna=False,observed=True).agg(n_cells=(\'matrix_row\',\'size\'),n_inner_cells=(\'inner_evaluation_eligible\',\'sum\')).reset_index()\n    fine.to_csv(out/\'fine_condition_coverage.csv\',index=False)\n    wellkeys=[k for k in [\'replicate\',\'cell_line\',\'library_id\',\'hash_plate\',\'hash_label_well\'] if k in cells]\n    wells=cell_roles.loc[eligible].groupby(wellkeys,dropna=False,observed=True).agg(n_outer_cells=(\'matrix_row\',\'size\'),n_inner_cells=(\'inner_evaluation_eligible\',\'sum\')).reset_index()\n    wells[\'unit_status\']=\'metadata composite identifiers; not verified independent biological units\'\n    wells.to_csv(out/\'well_scope.csv\',index=False)\n    directions=[{\'scenario\':\'B\',\'development_replicate\':a,\'evaluation_replicate\':b,\'status\':\'BLOCKED_PENDING_REPLICATE_PROVENANCE\',\'reason\':\'Step 1 biological replication unresolved\',\n                 \'gene_selection\':\'fit on development replicate only; do not reuse global-A outer genes\',\'evaluation_A_role\':\'predefined unsupervised graph/weights only; never evaluation B tuning\',\'preprocessing_and_mapping\':\'freeze development-only rules before evaluation\',\'independent_direction_count\':False} for a,b in [(REPS[0],REPS[1]),(REPS[1],REPS[0])]]\n    save_json(out/\'scenario_B_directions.json\',directions);pd.DataFrame(directions).to_csv(out/\'scenario_B_directions.csv\',index=False)\n    scenario={\'scenario\':\'A\',\'status\':\'data_protocol_prepared_NOT_MODELS_FITTED\',\'replicates\':REPS,\'scope\':\'two separate within-label analyses; report both; no cross-replicate generalization claim\',\n              \'cell_overlap\':\'same cells across molecular roles by design; no disjoint-cell claim\',\'outer_pair_contract\':str(run/\'A_preprocessing/evaluation_pair_contract.json\'),\'outer_metric_contract\':str(run/\'evaluation/metric_contract.json\'),\n              \'development_input\':\'A_train = Binomial(A,inner_q), all raw genes retained\',\'development_target\':\'A_validation = A - A_train; parameter selection only\',\n              \'inner_preprocessing\':\'per replicate: inner train only; all-gene library normalization; PPI-supported unique gene symbols ranked by log variance, detection, gene_idx\',\n              \'inner_and_outer_genes\':\'inner train-selected universe can differ from final full-A universe; tune pipeline rule, then refit once on full A using frozen outer genes; never select gene universe by final B\',\n              \'inner_mu_mapping\':\'mu_validation=expm1(max(log_prediction,0))*all_gene_L_train/scale*(1-inner_q)/inner_q\',\n              \'inner_zero_libraries\':\'exclude only zero training libraries from inner I; zero validation libraries retain all pairs; outer I unchanged\',\n              \'inner_secondary\':\'target=log1p(scale*A_validation/all_gene_L_validation); zero validation library gives zero target; target library never calibrates predictions\',\n              \'mu_floor\':metric[\'mu_floor\'],\'normalization_scale\':pair[\'normalization_scale\'],\'inner_q\':cfg[\'inner_q\'],\'outer_q\':pair[\'q\'],\n              \'final_refit\':\'freeze chosen parameters, then refit normalization/graphs/metacells/weights on full A separately per replicate; final B only once for scoring after all methods registered\',\n              \'hyperparameters\':\'selection not performed in Step 4; grids/methods must be fixed in Step 5 before opening development targets\',\n              \'metacells\':\'assignment from input fold only; aggregate target by frozen assignment; no target-based regrouping\',\n              \'gene_selection_scope_outer\':\'Step 2 gene universe selected using all eligible full-A cells across both labels; transductive within-cell setting, unsuitable for Scenario B\',\n              \'contexts\':\'keep full 15-context roster per label, including empty rows; no target-based selection\',\n              \'random_controls\':\'20 seeds initially; extend important results to 100 in Task 2; do not run random controls in Step 4\',\n              \'limitations\':[\'biological independence unresolved\',\'Poisson thinning independence is distribution-dependent\',\'original C-based population conditioning persists\',\'internal gene selection and outer gene selection use different scopes; disclose both\']}\n    save_json(out/\'scenario_A_contract.json\',scenario)\n    accesses=[[\'raw C\',\'forbidden\',\'original population already conditioned on C\'],[\'final B / library_B\',\'forbidden\',\'not opened in Step 4\'],[\'outer A\',\'allowed\',\'nested split; final refit later\'],[\'A_train\',\'allowed for fitting\',\'only source for inner normalization/genes/graphs/weights\'],[\'A_validation\',\'development scoring only\',\'no model fitting/weight updates\'],[\'metadata/PPI reference\',\'allowed\',\'fixed annotations; no target-based context filtering\']]\n    pd.DataFrame(accesses,columns=[\'source\',\'role\',\'restriction\']).to_csv(out/\'information_flow.csv\',index=False)\n    environment=save_environment(root,out)\n    save_json(out/\'output_manifest.json\',{\'files\':[{\'name\':p.name,\'sha256\':digest(p)} for p in sorted(out.iterdir()) if p.is_file() and p.name not in [\'output_manifest.json\',\'step4_summary.json\']]})\n    summary={\'status\':\'step4_prepared\',\'run_path\':str(run),\'development_path\':str(out),\'n_outer_cells\':int(eligible.sum()),\'n_outer_genes\':len(genes),\'n_outer_pairs\':int(eligible.sum())*len(genes),\n             \'n_inner_cells\':int(inner_mask.sum()),\'n_inner_pairs\':int(inner_mask.sum())*cfg[\'inner_n_genes\'],\'outer_eligible_inner_train_zero\':int((eligible & (trainlib==0)).sum()),\n             \'inner_genes_by_replicate\':summaries,\'scenario_A\':\'prepared\',\'scenario_B\':\'blocked_pending_replicate_provenance\',\'final_B_accessed\':False,\'models_fitted\':False,\'hyperparameters_selected\':False,\n             \'metric_contract_sha256_before\':identity[\'metric_contract_sha256\'],\'metric_contract_sha256_after\':digest(run/\'evaluation/metric_contract.json\'),\'engine_sha256\':ENGINE_SHA}\n    if summary[\'metric_contract_sha256_before\']!=summary[\'metric_contract_sha256_after\']:raise AssertionError(\'Step 3 contract changed\')\n    save_json(out/\'step4_summary.json\',summary)\n    save_json(root/\'results/raw\'/f"step4_{cfg[\'development_name\']}_summary.json",summary)\n    save_json(root/\'configs\'/f"step4_{cfg[\'development_name\']}.json",cfg)\n    return out,summary\n\ndef load_inner_input(development_path,replicate,block):\n    """Fitting interface exposes training counts only; no validation/final target."""\n    if replicate not in REPS:raise ValueError(\'Unknown replicate\')\n    out=Path(development_path);identity=read_json(out/\'identity.json\');run=Path(identity[\'run_path\'])\n    if (run/\'evaluation/target_access_lock.json\').exists():raise ValueError(\'Final B already accessed: further tuning is forbidden\')\n    folder=out/f\'block_{block:05d}\';rec=read_json(folder/\'receipt.json\');p=folder/\'A_train.npz\'\n    expected=next(x[\'sha256\'] for x in rec[\'files\'] if x[\'name\']==p.name)\n    if digest(p)!=expected:raise ValueError(\'Training cache damaged\')\n    roles=pd.read_parquet(out/\'cell_roles.parquet\');size=read_json(run/\'identity.json\')[\'cells_per_block\']\n    genes=pd.read_parquet(out/f\'{replicate}_inner_selected_genes.parquet\')\n    summary=read_json(out/\'step4_summary.json\');expected_genes=next(x[\'selected_genes_sha256\'] for x in summary[\'inner_genes_by_replicate\'] if x[\'replicate\']==replicate)\n    if digest(out/f\'{replicate}_inner_selected_genes.parquet\')!=expected_genes:raise ValueError(\'Inner gene file changed\')\n    a=sp.load_npz(p);meta=roles.iloc[block*size:block*size+a.shape[0]].copy()\n    keep=meta.replicate.eq(replicate).to_numpy() & meta.inner_evaluation_eligible.to_numpy(dtype=bool)\n    lib=np.asarray(a.sum(axis=1)).ravel()[keep];selected=a[keep][:,genes.gene_col.to_numpy(dtype=int)]\n    scale=read_json(out/\'scenario_A_contract.json\')[\'normalization_scale\']\n    y=(sp.diags(scale/lib)@selected.astype(float)).tocsr();y.data=np.log1p(y.data)\n    return {\'counts\':selected,\'log_normalized\':y,\'all_gene_library\':lib,\'matrix_row\':meta.loc[keep,\'matrix_row\'].to_numpy(),\'gene_idx\':genes.gene_idx.to_numpy(),\'replicate\':replicate}\n'
source_path = ROOT / "src/development_split.py"
source_path.parent.mkdir(parents=True, exist_ok=True)
source_bytes = SOURCE.encode("utf-8")
if source_path.exists() and source_path.read_bytes() != source_bytes:
    previous_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
    backup = source_path.with_name(f"development_split.{previous_sha[:12]}.backup.py")
    if not backup.exists():
        backup.write_bytes(source_path.read_bytes())
source_path.write_bytes(source_bytes)
spec = importlib.util.spec_from_file_location("development_split", source_path)
dev = importlib.util.module_from_spec(spec)
sys.modules["development_split"] = dev
spec.loader.exec_module(dev)
print("Step 4 source SHA256:", dev.ENGINE_SHA)


Step 4 source SHA256: a9ba1787282c70ef77e354e3e41eda93b988aae6e19e5d20c8a6823bc8ed355d


In [3]:
TEST_SOURCE = "from pathlib import Path\nimport sys,tempfile,json\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nfrom unittest.mock import patch\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/'src'))\nimport development_split as d\n\ndef rejects(fn):\n    try:fn()\n    except (ValueError,AssertionError):return\n    raise AssertionError('Expected rejection')\n\ndef fixture(root):\n    run=root/'data/splits/step2_fixture';prep=run/'A_preprocessing';prep.mkdir(parents=True)\n    (run/'evaluation').mkdir();(root/'data/metadata_audit').mkdir()\n    rng=np.random.default_rng(81);a=rng.poisson(15,size=(120,8)).astype(np.int64)\n    a[0]=0;a[0,0]=1;a[1]=0;a[1,1]=1\n    cells=pd.DataFrame({'matrix_row':np.arange(120),'cell_barcode':[f'cell{i}' for i in range(120)],\n                        'evaluation_eligible':True,'replicate':np.repeat(d.REPS,60),\n                        'cell_line':np.tile(np.repeat(d.LINES,20),2),'drug':np.tile(np.repeat(d.DRUGS,4),6),\n                        'dose_raw':1,'sgRNA_raw':'sg1','target_raw':'g1','hash_plate':'plate1','hash_label_well':'A1','library_id':'lib1'})\n    # Test NA metadata is retained, not silently dropped.\n    cells.loc[3,'sgRNA_raw']=None\n    genes=pd.DataFrame({'gene_col':np.arange(8),'gene_idx':np.arange(1,9),'gene_id':[f'id{i}' for i in range(8)],'gene_symbol':[f'g{i}' for i in range(8)]})\n    genes.to_parquet(run/'genes_all.parquet',index=False)\n    selected=genes.iloc[:4].copy();selected['selected_col']=np.arange(4)\n    selected.to_parquet(prep/'selected_genes_A.parquet',index=False);cells.to_parquet(prep/'evaluation_cells.parquet',index=False)\n    mapping=genes.copy();mapping['ppi_edge_supported']=True;mapping.to_csv(root/'data/metadata_audit/ppi_gene_mapping.csv',index=False)\n    pair={'cells_file':'evaluation_cells.parquet','genes_file':'selected_genes_A.parquet','cells_sha256':d.digest(prep/'evaluation_cells.parquet'),'genes_sha256':d.digest(prep/'selected_genes_A.parquet'),\n          'n_cells':120,'n_genes':4,'n_pairs':480,'q':.5,'normalization_scale':10000}\n    d.save_json(prep/'evaluation_pair_contract.json',pair)\n    metric={**pair,'pair_contract_sha256':d.digest(prep/'evaluation_pair_contract.json'),'mu_floor':1e-8}\n    d.save_json(run/'evaluation/metric_contract.json',metric);d.save_json(run/'identity.json',{'q':.5,'cells_per_block':40})\n    for b in range(3):\n        folder=run/f'block_{b:05d}';folder.mkdir();sp.save_npz(folder/'A_input.npz',sp.csr_matrix(a[b*40:(b+1)*40]))\n        # Poisoned targets must not be opened, even for source verification.\n        (folder/'B_target.npz').write_bytes(b'INVALID FINAL TARGET');(folder/'C_counts.npz').write_bytes(b'INVALID C')\n        d.save_json(folder/'receipt.json',{'files':[{'name':'A_input.npz','sha256':d.digest(folder/'A_input.npz')}]})\n    cfg={'project_root':str(root),'run_path':str(run),'development_name':'inner_q05_seed20261008_v1','inner_q':.5,'seed':20261008,'inner_n_genes':3}\n    return cfg,a\n\ndef tests():\n    checks=[]\n    for a in [np.array([[0,1,2],[100,0,7]]),np.zeros((2,3),dtype=int),np.array([[1]])]:\n        for q in [.01,.5,.99]:\n            tr,va=d.split_inner(a,q,1,0);assert np.array_equal((tr+va).toarray(),a)\n            again=d.split_inner(a,q,1,0);assert (tr-again[0]).nnz==0\n    checks.append('integer conservation, sparse zeros, deterministic seeds, near-boundary q')\n    for q in [0,1,float('nan'),-1]:rejects(lambda:d.split_inner([[1]],q,1,0))\n    for a in [[[-1]],[[.5]],[[float('nan')]]]:rejects(lambda:d.split_inner(a,.5,1,0))\n    checks.append('invalid counts and configuration rejected')\n    raw=pd.DataFrame({'gene_col':[0,1,2],'gene_idx':[1,2,3],'gene_symbol':['same','same','other']})\n    stats=[2,np.array([2.,2.,2.]),np.array([4.,4.,4.]),np.array([1,1,1])]\n    selected,_=d.rank_genes(raw,np.ones(3,dtype=bool),stats,2)\n    assert selected.gene_idx.tolist()==[1,3]\n    rejects(lambda:d.rank_genes(raw,np.ones(3,dtype=bool),stats,3))\n    checks.append('deterministic gene ties, unique symbols, insufficient genes rejected')\n    with tempfile.TemporaryDirectory() as t:\n        root=Path(t);cfg,a=fixture(root)\n        original_load=sp.load_npz\n        def safe_load(path,*args,**kwargs):\n            assert Path(path).name not in ['B_target.npz','C_counts.npz']\n            return original_load(path,*args,**kwargs)\n        with patch.object(sp,'load_npz',safe_load):out,s=d.prepare_development(cfg)\n        assert s['n_outer_cells']==120 and s['n_outer_pairs']==480 and not s['final_B_accessed']\n        assert s['metric_contract_sha256_before']==s['metric_contract_sha256_after']\n        roles=pd.read_parquet(out/'cell_roles.parquet')\n        expected=0\n        for b in range(3):\n            tr=sp.load_npz(out/f'block_{b:05d}/A_train.npz');va=sp.load_npz(out/f'block_{b:05d}/A_validation.npz')\n            assert np.array_equal((tr+va).toarray(),a[b*40:(b+1)*40]);expected+=int(np.asarray(tr.sum(axis=1)).ravel().sum())\n        assert expected==int(roles.library_A_train.sum())\n        assert len(pd.read_csv(out/'outer_coverage.csv'))==30\n        assert int(pd.read_csv(out/'outer_coverage.csv').n_pairs.sum())==480\n        assert all(r['status'].startswith('BLOCKED') for r in d.read_json(out/'scenario_B_directions.json'))\n        checks.append('end-to-end, both labels/all contexts, poisoned B/C never opened, outer contract unchanged')\n        # Independent dense oracle for inner training preprocessing.\n        for rep in d.REPS:\n            keep=roles.replicate.eq(rep).to_numpy() & roles.inner_evaluation_eligible.to_numpy()\n            alltrain=np.vstack([sp.load_npz(out/f'block_{b:05d}/A_train.npz').toarray() for b in range(3)])[keep]\n            y=np.log1p(10000*alltrain/alltrain.sum(axis=1,keepdims=True))\n            table=pd.read_parquet(out/f'{rep}_inner_gene_stats.parquet')\n            assert np.allclose(table.train_log_variance,y.var(axis=0,ddof=1))\n            assert np.allclose(table.train_log_mean,y.mean(axis=0))\n        checks.append('train-only gene statistics match independent dense oracle')\n        # Inner validation-zero cells remain in common I if their train library is positive.\n        assert (roles.inner_evaluation_eligible==(roles.evaluation_eligible & (roles.library_A_train>0))).all()\n        assert (roles.evaluation_eligible & (roles.library_A_train==0)).any()\n        assert (roles.inner_evaluation_eligible & (roles.library_A_validation==0)).any()\n        checks.append('zero training excluded internally only; zero validation never filters I')\n        target_hash=d.digest(out/'block_00000/A_validation.npz')\n        def training_only(path,*args,**kwargs):\n            assert Path(path).name=='A_train.npz'\n            return original_load(path,*args,**kwargs)\n        with patch.object(sp,'load_npz',training_only):inp=d.load_inner_input(out,d.REPS[0],0)\n        assert inp['counts'].shape[1]==3 and np.all(inp['all_gene_library']>0)\n        assert d.digest(out/'block_00000/A_validation.npz')==target_hash\n        checks.append('fitting interface never opens any validation target')\n        with patch.object(sp,'load_npz',safe_load):out2,s2=d.prepare_development(cfg)\n        assert s==s2\n        rejects(lambda:d.prepare_development({**cfg,'inner_q':.6}))\n        checks.append('verified cache replay identical; frozen parameter changes rejected')\n        # Corrupted cache and final-target-access guard.\n        p=out/'block_00000/A_validation.npz';saved=p.read_bytes();p.write_bytes(b'corrupt')\n        rejects(lambda:d.prepare_development(cfg));p.write_bytes(saved)\n        d.save_json(Path(cfg['run_path'])/'evaluation/target_access_lock.json',{'accessed':True})\n        rejects(lambda:d.prepare_development(cfg));rejects(lambda:d.load_inner_input(out,d.REPS[0],0))\n        checks.append('cache damage and prior final-B access rejected')\n    return {'status':'PASS','checks':checks}\n\nif __name__=='__main__':print(json.dumps(tests(),ensure_ascii=False,indent=2))\n"
test_namespace = {
    "__name__": "step4_embedded_tests",
    "__file__": str(ROOT / "tests/test_development_split.py"),
}
exec(compile(TEST_SOURCE, "embedded_step4_tests.py", "exec"), test_namespace)
TEST_REPORT = test_namespace["tests"]()
assert TEST_REPORT["status"] == "PASS"
print(json.dumps(TEST_REPORT, ensure_ascii=False, indent=2))


Step 4 inner split 1/3
Step 4 inner split 2/3
Step 4 inner split 3/3
Step 4 inner split 1/3
Step 4 inner split 2/3
Step 4 inner split 3/3
{
  "status": "PASS",
  "checks": [
    "integer conservation, sparse zeros, deterministic seeds, near-boundary q",
    "invalid counts and configuration rejected",
    "deterministic gene ties, unique symbols, insufficient genes rejected",
    "end-to-end, both labels/all contexts, poisoned B/C never opened, outer contract unchanged",
    "train-only gene statistics match independent dense oracle",
    "zero training excluded internally only; zero validation never filters I",
    "fitting interface never opens any validation target",
    "verified cache replay identical; frozen parameter changes rejected",
    "cache damage and prior final-B access rejected"
  ]
}


In [4]:
inputs = dev.load_inputs(CONFIG)
root, run, frozen_cells, frozen_genes, raw_genes, ppi_support, pair, metric, identity = inputs

print("Final evaluation cells:", pair["n_cells"])
print("Final evaluation genes:", pair["n_genes"])
print("Final evaluation pairs:", pair["n_pairs"])

assert pair["n_cells"] * pair["n_genes"] == pair["n_pairs"]

print("Final evaluation contract remains in place:", run / "evaluation/metric_contract.json")

del inputs, frozen_cells, frozen_genes, raw_genes, ppi_support

Final evaluation cells: 989299
Final evaluation genes: 2500
Final evaluation pairs: 2473247500
Final evaluation contract remains in place: /home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/evaluation/metric_contract.json


In [5]:
assert TEST_REPORT["status"] == "PASS"
DEVELOPMENT_PATH, STEP4_SUMMARY = dev.prepare_development(CONFIG)
dev.save_json(DEVELOPMENT_PATH / "synthetic_test_report.json", TEST_REPORT)
manifest = dev.read_json(DEVELOPMENT_PATH / "output_manifest.json")
manifest["files"] = [x for x in manifest["files"] if x["name"] != "synthetic_test_report.json"]
manifest["files"].append({"name": "synthetic_test_report.json", "sha256": dev.digest(DEVELOPMENT_PATH / "synthetic_test_report.json")})
dev.save_json(DEVELOPMENT_PATH / "output_manifest.json", manifest)
print(json.dumps(STEP4_SUMMARY, ensure_ascii=False, indent=2))


Step 4 inner split 1/211
Step 4 inner split 2/211
Step 4 inner split 3/211
Step 4 inner split 4/211
Step 4 inner split 5/211
Step 4 inner split 6/211
Step 4 inner split 7/211
Step 4 inner split 8/211
Step 4 inner split 9/211
Step 4 inner split 10/211
Step 4 inner split 11/211
Step 4 inner split 12/211
Step 4 inner split 13/211
Step 4 inner split 14/211
Step 4 inner split 15/211
Step 4 inner split 16/211
Step 4 inner split 17/211
Step 4 inner split 18/211
Step 4 inner split 19/211
Step 4 inner split 20/211
Step 4 inner split 21/211
Step 4 inner split 22/211
Step 4 inner split 23/211
Step 4 inner split 24/211
Step 4 inner split 25/211
Step 4 inner split 26/211
Step 4 inner split 27/211
Step 4 inner split 28/211
Step 4 inner split 29/211
Step 4 inner split 30/211
Step 4 inner split 31/211
Step 4 inner split 32/211
Step 4 inner split 33/211
Step 4 inner split 34/211
Step 4 inner split 35/211
Step 4 inner split 36/211
Step 4 inner split 37/211
Step 4 inner split 38/211
Step 4 inner split 39

In [6]:
outer_coverage = pd.read_csv(DEVELOPMENT_PATH / "outer_coverage.csv")
inner_coverage = pd.read_csv(DEVELOPMENT_PATH / "inner_coverage.csv")

assert int(outer_coverage.n_pairs.sum()) == STEP4_SUMMARY["n_outer_pairs"]
assert int(inner_coverage.n_pairs.sum()) == STEP4_SUMMARY["n_inner_pairs"]
assert len(outer_coverage) == len(dev.REPS) * len(dev.LINES) * len(dev.DRUGS)

print("Final A/B coverage:")
print(outer_coverage.to_string(index=False))

print("Internal T/V coverage:")
print(inner_coverage.to_string(index=False))

print(
    "Cells in final I excluded from internal tuning because T library = 0:",
    STEP4_SUMMARY["outer_eligible_inner_train_zero"],
)

Final A/B coverage:
  replicate cell_line       drug  n_cells  n_genes   n_pairs
replicate_1      A172    Vehicle    33460     2500  83650000
replicate_1      A172  Lapatinib    47258     2500 118145000
replicate_1      A172 Nintedanib    36307     2500  90767500
replicate_1      A172 Trametinib    13723     2500  34307500
replicate_1      A172    ZSTK474    34561     2500  86402500
replicate_1      T98G    Vehicle    36802     2500  92005000
replicate_1      T98G  Lapatinib    67012     2500 167530000
replicate_1      T98G Nintedanib    31554     2500  78885000
replicate_1      T98G Trametinib    25943     2500  64857500
replicate_1      T98G    ZSTK474    32803     2500  82007500
replicate_1     U87MG    Vehicle    25839     2500  64597500
replicate_1     U87MG  Lapatinib    41330     2500 103325000
replicate_1     U87MG Nintedanib    35154     2500  87885000
replicate_1     U87MG Trametinib    11412     2500  28530000
replicate_1     U87MG    ZSTK474    17235     2500  43087500
repl

In [7]:
scenario_A = dev.read_json(DEVELOPMENT_PATH / "scenario_A_contract.json")
scenario_B = dev.read_json(DEVELOPMENT_PATH / "scenario_B_directions.json")
assert all(x["status"] == "BLOCKED_PENDING_REPLICATE_PROVENANCE" for x in scenario_B)
print(json.dumps(scenario_A, ensure_ascii=False, indent=2))
print(pd.DataFrame(scenario_B)[["development_replicate", "evaluation_replicate", "status"]].to_string(index=False))
print(pd.read_csv(DEVELOPMENT_PATH / "information_flow.csv").to_string(index=False))


{
  "scenario": "A",
  "status": "data_protocol_prepared_NOT_MODELS_FITTED",
  "replicates": [
    "replicate_1",
    "replicate_2"
  ],
  "scope": "two separate within-label analyses; report both; no cross-replicate generalization claim",
  "cell_overlap": "same cells across molecular roles by design; no disjoint-cell claim",
  "outer_pair_contract": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/A_preprocessing/evaluation_pair_contract.json",
  "outer_metric_contract": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/evaluation/metric_contract.json",
  "development_input": "A_train = Binomial(A,inner_q), all raw genes retained",
  "development_target": "A_validation = A - A_train; parameter selection only",
  "inner_preprocessing": "per replicate: inner train only; all-gene library normalization; PPI-supported unique gene symbols ranked by log variance, detection, gene_idx",
  "inner_and_outer_genes": "inner tr

In [8]:
assert STEP4_SUMMARY["final_B_accessed"] is False
assert STEP4_SUMMARY["models_fitted"] is False
assert STEP4_SUMMARY["hyperparameters_selected"] is False
assert STEP4_SUMMARY["metric_contract_sha256_before"] == dev.digest(
    run / "evaluation/metric_contract.json"
)
assert not (run / "evaluation/target_access_lock.json").exists()

print(json.dumps(dev.read_json(DEVELOPMENT_PATH / "environment.json"), ensure_ascii=False, indent=2))
print("Step 4 data and scenario preparation is complete; final B was not accessed.")
print("Output directory:", DEVELOPMENT_PATH)
print("Please send back:")

for name in [
    "step4_summary.json",
    "synthetic_test_report.json",
    "outer_coverage.csv",
    "inner_coverage.csv",
    "scenario_A_contract.json",
    "scenario_B_directions.json",
]:
    print(DEVELOPMENT_PATH / name)

{
  "python": "3.11.18 (main, Oct 26 2026, 18:19:35) [GCC 14.3.0]",
  "executable": "/environment/miniconda3/envs/sciplex-dualgraph/bin/python",
  "platform": "Linux-5.15.0-105-generic-x86_64-with-glibc2.35",
  "numpy": "2.4.6",
  "pandas": "3.1.0",
  "scipy": "1.17.1",
  "engine_sha256": "a9ba1787282c70ef77e354e3e41eda93b988aae6e19e5d20c8a6823bc8ed355d",
  "git": {
    "commit": null,
    "status": "not_a_git_checkout"
  }
}
Step 4 data and scenario preparation is complete; final B was not accessed.
Output directory: /home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/development/inner_q05_seed20261008
Please send back:
/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/development/inner_q05_seed20261008/step4_summary.json
/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/development/inner_q05_seed20261008/synthetic_test_report.json
/home/featurize/work/biological_prior_validation/data/s